# 009 Custom Middleware

这是 LangChain 学习线的第九份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/middleware/custom

学习目标：

1. 理解 custom middleware 在 agent 运行链路里的位置
2. 学会 decorator-based middleware 和 class-based middleware 两种写法
3. 看懂 `before_agent`、`before_model`、`after_model`、`after_agent`
4. 看懂 `wrap_model_call` 和 `wrap_tool_call`
5. 学会给 middleware 增加自定义 state
6. 理解 agent jump 为什么是强控制能力
7. 对比本仓库 Harness runtime 里 ledger、approval、context、verification 应该放在哪里

这一讲全部使用 fake model，不消耗真实模型额度。

## 1. Middleware 的心智模型

如果用 Java 来类比：

```text
LangChain Middleware
  ~= Spring HandlerInterceptor / Servlet Filter / AOP Around Advice
```

区别是：LangChain middleware 不是拦截 HTTP 请求，而是拦截 agent 的内部节点。

它可以在这些位置工作：

| Hook | 触发时机 | 常见用途 |
| --- | --- | --- |
| `before_agent` | agent 开始前，本次 invocation 只触发一次 | 初始化、审计、输入校验 |
| `before_model` | 每次模型调用前 | 动态 prompt、上下文裁剪、停止条件 |
| `after_model` | 每次模型响应后 | 记录响应、检查 tool call、统计 token |
| `after_agent` | agent 完成后，本次 invocation 只触发一次 | 收尾、审计、输出整理 |
| `wrap_model_call` | 包住每次模型调用 | 重试、fallback、动态换模型、耗时监控 |
| `wrap_tool_call` | 包住每次工具调用 | 权限、重试、审计、错误包装 |

本仓库 Harness 里很多能力都可以类比到 middleware：

- ledger：适合放在 `before_*` / `after_*`
- approval：适合放在 `wrap_tool_call`
- context compact：适合放在 `before_model`
- recovery：适合放在 `wrap_model_call` / `wrap_tool_call`
- final synthesis：通常不应该完全交给 middleware，而应该由 coordinator / service 明确控制。

In [10]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    ModelRequest,
    ModelResponse,
    after_agent,
    after_model,
    before_agent,
    before_model,
    wrap_model_call,
    wrap_tool_call,
)
from langchain_core.language_models.fake_chat_models import FakeListChatModel, FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.runtime import Runtime


## 2. Decorator 写法：最小日志 middleware

Decorator 写法适合小而清晰的 middleware。

下面这个例子只做日志，不修改任何状态。你可以把它理解成 Java 里的一个轻量拦截器。

In [2]:
@before_agent
def log_before_agent(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print("[before_agent] messages=", len(state.get("messages", [])))
    return None


@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print("[before_model] messages=", len(state.get("messages", [])))
    return None


@after_model
def log_after_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print("[after_model] messages=", len(state.get("messages", [])))
    return None


@after_agent
def log_after_agent(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print("[after_agent] messages=", len(state.get("messages", [])))
    return None


In [3]:
log_agent = create_agent(
    model=FakeListChatModel(responses=["这是 fake model 的回答。"]),
    tools=[],
    middleware=[log_before_agent, log_before_model, log_after_model, log_after_agent],
)

log_result = log_agent.invoke({"messages": [{"role": "user", "content": "你好"}]})
print("final:", log_result["messages"][-1].content)


[before_agent] messages= 1
[before_model] messages= 1
[after_model] messages= 2
[after_agent] messages= 2
final: 这是 fake model 的回答。


## 3. `wrap_model_call`：包住模型调用

`before_model` 和 `after_model` 是在固定点执行。

`wrap_model_call` 更像 Java AOP 的 around advice：

```text
before
  -> handler(request)
after
```

它适合做：

- 模型调用耗时统计
- 动态换模型
- fallback
- 统一错误包装
- 模型请求审计

In [4]:
@wrap_model_call
def monitor_model_call(request: ModelRequest, handler) -> ModelResponse:
    print("[wrap_model_call] before model, messages=", len(request.messages))
    response = handler(request)
    print("[wrap_model_call] after model, response_messages=", len(response.result))
    return response


model_monitor_agent = create_agent(
    model=FakeListChatModel(responses=["模型调用被 middleware 包住了。"]),
    tools=[],
    middleware=[monitor_model_call],
)

model_monitor_result = model_monitor_agent.invoke({"messages": [{"role": "user", "content": "测试模型 hook"}]})
print("final:", model_monitor_result["messages"][-1].content)


[wrap_model_call] before model, messages= 1
[wrap_model_call] after model, response_messages= 1
final: 模型调用被 middleware 包住了。


## 4. `wrap_tool_call`：包住工具调用

`wrap_tool_call` 是最像 Harness approval 的位置。

原因很简单：权限判断应该发生在工具真正执行前，而不是模型生成 tool call 之后就默认执行。

下面用 fake model 模拟一次 tool call。为了让 fake model 支持工具绑定，这里做一个很薄的子类。

In [5]:
class ToolCallingFakeModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self


@tool
def add_one(number: int) -> str:
    """Add one to a number."""
    return str(number + 1)


@wrap_tool_call
def monitor_tool_call(request, handler):
    print("[wrap_tool_call] tool=", request.tool_call["name"], "args=", request.tool_call["args"])
    result = handler(request)
    print("[wrap_tool_call] result=", result.content)
    return result


tool_calling_model = ToolCallingFakeModel(
    responses=[
        AIMessage(content="", tool_calls=[{"name": "add_one", "args": {"number": 41}, "id": "call_1"}]),
        AIMessage(content="工具返回 42。"),
    ]
)

tool_monitor_agent = create_agent(
    model=tool_calling_model,
    tools=[add_one],
    middleware=[monitor_tool_call],
)

tool_monitor_result = tool_monitor_agent.invoke({"messages": [{"role": "user", "content": "41 加 1 等于多少？"}]})

for message in tool_monitor_result["messages"]:
    print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))


[wrap_tool_call] tool= add_one args= {'number': 41}
[wrap_tool_call] result= 42
human 41 加 1 等于多少？
ai 
tool 42
ai 工具返回 42。


## 5. Class 写法：适合一组相关 hook

当一个 middleware 需要多个 hook、内部配置或复用状态时，用 class 更清晰。

这和 Java 里写一个实现类很像：

```java
class TraceInterceptor implements HandlerInterceptor { ... }
```

LangChain 里继承 `AgentMiddleware`。

In [6]:
class TraceMiddleware(AgentMiddleware):
    def before_agent(self, state, runtime):
        print("[class before_agent] start")
        return None

    def before_model(self, state, runtime):
        print("[class before_model] messages=", len(state.get("messages", [])))
        return None

    def after_agent(self, state, runtime):
        print("[class after_agent] done")
        return None


class_agent = create_agent(
    model=FakeListChatModel(responses=["class middleware 已执行。"]),
    tools=[],
    middleware=[TraceMiddleware()],
)

class_result = class_agent.invoke({"messages": [{"role": "user", "content": "测试 class middleware"}]})
print("final:", class_result["messages"][-1].content)


[class before_agent] start
[class before_model] messages= 1
[class after_agent] done
final: class middleware 已执行。


## 6. 自定义 State Schema

middleware 不只能看 `messages`，也可以给 agent state 增加自己的字段。

这适合做：

- 计数器
- 审计字段
- 风险等级
- session 元数据
- 本仓库里的 ledger 摘要字段

下面给 state 加一个 `model_call_count`。

In [7]:
class StudyState(AgentState):
    model_call_count: int


@before_model(state_schema=StudyState)
def count_model_calls(state: StudyState, runtime: Runtime) -> dict[str, Any]:
    current = state.get("model_call_count", 0) + 1
    print("model_call_count=", current)
    return {"model_call_count": current}


state_agent = create_agent(
    model=FakeListChatModel(responses=["state 已更新。"]),
    tools=[],
    middleware=[count_model_calls],
)

state_result = state_agent.invoke({"messages": [{"role": "user", "content": "测试 state"}]})
print("result keys:", state_result.keys())
print("model_call_count:", state_result.get("model_call_count"))


model_call_count= 1
result keys: dict_keys(['messages', 'model_call_count'])
model_call_count: 1


## 7. Agent Jump：强控制能力

custom middleware 还可以让 agent 跳到指定节点，例如直接结束。

这是一种强控制能力，适合做：

- 预算耗尽时提前结束
- 风险过高时阻断执行
- 上下文过长时要求用户重开任务

注意：能 jump 的 hook 必须声明 `can_jump_to`。

In [8]:
@before_model(can_jump_to=["end"])
def stop_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    if len(state.get("messages", [])) >= 1:
        return {
            "messages": [AIMessage(content="被 middleware 提前结束，没有调用模型。")],
            "jump_to": "end",
        }
    return None


jump_agent = create_agent(
    model=FakeListChatModel(responses=["这句话不应该出现。"]),
    tools=[],
    middleware=[stop_before_model],
)

jump_result = jump_agent.invoke({"messages": [{"role": "user", "content": "请回答"}]})

for message in jump_result["messages"]:
    print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))


human 请回答
ai 被 middleware 提前结束，没有调用模型。


## 8. 执行顺序

一个最小 agent 调用大致会经历：

```text
before_agent
  before_model
    wrap_model_call
      model.invoke
    wrap_model_call returns
  after_model
  如果模型要求调用工具：
    wrap_tool_call
      tool.invoke
    wrap_tool_call returns
  可能再次 before_model / model / after_model
after_agent
```

这也是为什么权限审批更适合 `wrap_tool_call`，而不是 `after_model`。`after_model` 只能看到模型想调用工具，`wrap_tool_call` 才站在工具真正执行前。

## 9. 和本仓库 Harness 的对应关系

| Harness 关注点 | LangChain custom middleware 放置点 | 说明 |
| --- | --- | --- |
| ledger 记录 | `before_agent`、`before_model`、`after_model`、`after_agent` | 记录每一步发生了什么 |
| 工具审批 | `wrap_tool_call` | 工具执行前拦截 |
| 上下文裁剪 | `before_model` | 模型调用前整理 messages |
| 模型 fallback | `wrap_model_call` | 模型失败时换模型或重试 |
| 工具错误恢复 | `wrap_tool_call` | 捕获异常，返回可读错误 |
| 风险阻断 | `before_model` + `can_jump_to=["end"]` | 预算耗尽或风险过高时结束 |
| 最终综合 | 不建议完全放 middleware | coordinator / service 更适合显式综合 |

关键判断：middleware 适合做横切控制，不适合把业务主流程藏进去。

## 10. 最小练习

请你思考下面三个问题：

1. 如果要记录每次模型调用的输入消息数量，应该用哪个 hook？
2. 如果要禁止 `delete_file` 工具自动执行，应该用哪个 hook？
3. 如果上下文超过 50 条消息就直接结束，应该用哪个 hook 和哪个配置？

参考答案：

1. `before_model` 或 `wrap_model_call`
2. `wrap_tool_call`
3. `before_model(can_jump_to=["end"])`，返回 `{"jump_to": "end"}`

## 11. 本讲小结

这一讲的重点不是记 API 名字，而是建立判断：

```text
middleware 是 agent 内部运行链路的横切控制层。
```

你现在应该能判断：

- 日志和审计放在哪里
- 权限审批放在哪里
- 上下文治理放在哪里
- 模型/工具恢复放在哪里
- 哪些逻辑不应该塞进 middleware

下一步如果继续学习，可以把本仓库的 Harness approval 思路迁移成一个 LangChain `wrap_tool_call` middleware。